<a href="https://colab.research.google.com/github/palaceIA/ml-titanic/blob/main/svm_pratica_alunos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prática — SVM

**Problema:** prever quem sobreviveu ao **Titanic**.
Comparando o SVM com a Random Forest, que você mesmo(a) vai montar.


Rode as células **uma por uma, de cima para baixo** (`Shift + Enter`). As células com `___` só funcionam depois que você completar.

## 0. Preparação (já pronta, só rode)

In [ ]:
import pandas as pd
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(url)

# Limpeza e encoding (iguais à prática anterior)
df = df[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]].copy()
df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna("S")
df["sex"] = df["sex"].map({"male": 0, "female": 1})
df = pd.get_dummies(df, columns=["embarked"], dtype=int)

X = df.drop(columns="survived")
y = df["survived"]
print(f"{len(df)} passageiros | {y.mean():.0%} sobreviveram")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultados = []

def avaliar(nome, modelo):
    r = cross_validate(modelo, X, y, cv=cv, return_train_score=True)
    treino, valid = r["train_score"].mean(), r["test_score"].mean()
    resultados.append({"Modelo": nome, "Treino": treino, "Validação": valid})
    print(f"{nome:<40} treino: {treino:.1%}   validação: {valid:.1%}")

891 passageiros | 38% sobreviveram


---
## Etapa 0 — Sua Random Forest de referência

Antes do SVM, monte **você** a Random Forest da aula passada. Ela será a referência para comparar.

> **Lembrete:** a Random Forest treina muitas árvores diferentes e decide pelo voto da maioria.
> Hiperparâmetros principais: `n_estimators` (nº de árvores) e `max_depth` (profundidade de cada árvore).
> A classe se chama `RandomForestClassifier` (já importada).

**0.1** Escreva a linha completa do modelo: **300 árvores** e **profundidade máxima 6**.

In [ ]:
avaliar("Random Forest (referência)",RandomForestClassifier(n_estimators=300,max_depth=6))

Random Forest (referência)               treino: 87.6%   validação: 82.9%


<details><summary> Travou? Clique para ver uma dica</summary>

O formato é `NomeDoModelo(parametro=valor, parametro=valor)`. Use `n_estimators=300`, `max_depth=6`
e acrescente `random_state=42` para o resultado ser reproduzível.
</details>

<details><summary>Ainda travado? Segunda dica</summary>

`RandomForestClassifier(n_estimators=300, max_depth=___, random_state=42)`: falta só um número.
</details>

---
## Etapa 1 — O SVM e a escala dos dados

> **Lembrete:** o SVM traça a fronteira com a **maior margem** entre as classes e decide por **distância**.
> Se uma coluna tem números muito maiores que as outras, ela domina a distância.
> O `StandardScaler` coloca todas as colunas na mesma escala, e o `make_pipeline` encadeia escala + modelo.

**1.1** Veja as escalas das colunas:

In [ ]:
X.describe().loc[["min", "max"]]

,pclass,sex,age,sibsp,parch,fare,embarked_C,embarked_Q,embarked_S
min,1.0,0.0,0.42,0.0,0.0,0.0000,0.0,0.0,0.0
max,3.0,1.0,80.00,8.0,6.0,512.3292,1.0,1.0,1.0


**1.2** Rode o SVM **sem** escalar os dados:

In [ ]:
avaliar("SVM sem escala", SVC())

SVM sem escala                           treino: 68.3%   validação: 67.8%


**1.3** Agora com escala. Complete o pipeline com o `StandardScaler()`.
Dica: na demo foi `make_pipeline(StandardScaler(), SVC())`.

In [ ]:
avaliar("SVM com escala", make_pipeline(StandardScaler(), SVC()))

SVM com escala                           treino: 84.1%   validação: 82.7%


**1.4 Quanto a validação mudou? Qual coluna você acha que atrapalhava o SVM sem escala, e por quê?**

*Sua resposta:*A validação aumentou de 67,8% para 82,7%, uma melhora de 14,9 pontos percentuais. A coluna fare provavelmente atrapalhava, pois seus valores são muito maiores que os das outras colunas, fazendo o SVM dar mais peso a essa variável. O StandardScaler colocou as variáveis em escalas semelhantes, melhorando o desempenho do modelo.

---
## Etapa 2 — O hiperparâmetro C

> **Lembrete:** `C` controla o quanto o SVM aceita errar no treino.
> **C baixo** → margem larga, tolera erros, modelo simples.
> **C alto** → margem estreita, tenta acertar tudo, risco de decorar (overfitting).
> O padrão é `C=1`.

**2.1** Teste C muito baixo e muito alto. Substitua por `0.01` e `100`:

In [ ]:
avaliar("SVM C=0.01", make_pipeline(StandardScaler(), SVC(C=0.01)))
avaliar("SVM C=100",  make_pipeline(StandardScaler(), SVC(C=100)))

SVM C=0.01                               treino: 61.6%   validação: 61.6%
SVM C=100                                treino: 88.0%   validação: 79.8%


**2.2 Com C=0.01, a validação ficou perto de 62%. Que número é esse? (Dica: quantos passageiros morreram?)**

*Sua resposta:*É aproximadamente a porcentagem de passageiros que morreram no Titanic (cerca de 62%). Com C muito baixo, o SVM fica simples demais e pode prever a classe majoritária para quase todos os passageiros.

**2.3 Com C=100, o que aconteceu com a distância entre treino e validação?**

*Sua resposta:*A diferença entre treino (88,0%) e validação (79,8%) aumentou para 8,2 pontos percentuais, indicando possível overfitting: o modelo se ajustou melhor aos dados de treino, mas teve desempenho inferior na validação.

---
## Etapa 3 — Kernel e gamma

> **Lembrete:** o **kernel** permite fronteiras curvas. `kernel="linear"` só traça retas (planos);
> `kernel="rbf"` (o padrão) faz curvas, como no exemplo dos círculos.
> **`gamma`** controla o alcance de cada ponto: gamma alto → fronteira tortuosa, com "ilhas" → overfitting.

**3.1** Compare o kernel linear com o rbf. Complete com `"linear"`:

In [ ]:
avaliar("SVM kernel linear", make_pipeline(StandardScaler(), SVC(kernel="linear")))

SVM kernel linear                        treino: 79.0%   validação: 78.7%


**3.2** Agora teste um gamma alto. Substitua por `10`:

In [ ]:
avaliar("SVM gamma=10", make_pipeline(StandardScaler(), SVC(gamma=10)))

SVM gamma=10                             treino: 91.2%   validação: 75.0%


**3.3 O kernel rbf (Etapa 1.3) foi melhor que o linear? O que isso sugere sobre a fronteira entre sobreviventes e não sobreviventes?**

*Sua resposta:*Sim, o kernel RBF foi melhor que o linear, indicando que a fronteira entre sobreviventes e não sobreviventes pode ser não linear.

**3.4 Com gamma=10, o que aconteceu com treino e validação?**

*Sua resposta:*Com gamma=10, o treino aumentou para 91,2%, mas a validação caiu para 75,0%. Isso indica overfitting, pois o modelo se ajustou demais aos dados de treino e generalizou pior.

---
## 🏆 Desafio — Busca automática de hiperparâmetros

Testar combinações na mão cansa. O `GridSearchCV` testa **todas** as combinações de uma grade
e devolve a melhor, usando validação cruzada.

Complete a grade com pelo menos 3 valores de `C` e 3 de `gamma`. Sugestão: `[0.1, 1, 10]` e `[0.01, 0.1, 1]`.
Atenção ao nome: dentro do pipeline, o parâmetro se chama `svc__C` (dois underlines).

In [ ]:
from sklearn.model_selection import GridSearchCV

grade = {
    "svc__C":    [0.1, 1, 10],
    "svc__gamma": [0.01, 0.1, 1],
}
busca = GridSearchCV(make_pipeline(StandardScaler(), SVC()), grade, cv=cv)
busca.fit(X, y)

print("Melhor combinação:", busca.best_params_)
print(f"Validação: {busca.best_score_:.1%}")

Melhor combinação: {'svc__C': 1, 'svc__gamma': 0.1}
Validação: 82.5%


**Desafio: o melhor SVM superou a sua Random Forest (Etapa 0)? Por quanto?**

*Sua resposta:*Não. O melhor SVM obteve 82,5% de validação, enquanto a Random Forest obteve 84,2%. Portanto, o SVM ficou 1,7 ponto percentual abaixo da Random Fores

---
## Seus resultados

In [ ]:
tabela = (pd.DataFrame(resultados)
          .drop_duplicates("Modelo", keep="last")
          .sort_values("Validação", ascending=False)
          .reset_index(drop=True))
tabela.style.format({"Treino": "{:.1%}", "Validação": "{:.1%}"})

,Modelo,Treino,Validação
0,Random Forest (referência),87.6%,82.9%
1,SVM com escala,84.1%,82.7%
2,SVM C=100,88.0%,79.8%
3,SVM kernel linear,79.0%,78.7%
4,SVM gamma=10,91.2%,75.0%
5,SVM sem escala,68.3%,67.8%
6,SVM C=0.01,61.6%,61.6%


**Se você fosse colocar um destes modelos em produção, qual escolheria? Justifique em uma frase.**

*Sua resposta:*Escolheria a Random Forest, pois obteve a maior validação (82,9%), apresentou bom equilíbrio entre treino e validação e tende a generalizar melhor que os modelos com sinais de overfitting.